# Legal Contract Clause Analyzer - Fine-Tuning Notebook
## Fine-tune Qwen2.5-3B on CUAD dataset with QLoRA + Unsloth

**What this notebook does:**
1. Installs Unsloth (fast fine-tuning library)
2. Loads Qwen2.5-3B model with 4-bit quantization
3. Adds LoRA adapters (trainable parameters)
4. Loads our formatted CUAD training data
5. Trains the model (~30-60 min on T4 GPU)
6. Tests the model on sample contracts
7. Saves the trained model to Google Drive

**Before running:** Upload `train.jsonl` and `val.jsonl` from your laptop to Google Drive.

---
## CELL 1: Install Unsloth
This installs the Unsloth library which makes training 2x faster and uses 60% less memory.

In [ ]:
%%capture
!pip install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps \
    git+https://github.com/unslothai/unsloth.git

---
## CELL 2: Check GPU
Verify we have a T4 GPU connected.

In [ ]:
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("ERROR: No GPU found! Go to Runtime > Change runtime type > T4 GPU")

---
## CELL 3: Mount Google Drive
This connects your Google Drive so we can load the training data and save the model.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Check if training data exists
import os
data_dir = '/content/drive/MyDrive/legal-contract-llm'
train_file = os.path.join(data_dir, 'train.jsonl')
val_file = os.path.join(data_dir, 'val.jsonl')

if os.path.exists(train_file):
    size_mb = os.path.getsize(train_file) / (1024*1024)
    print(f"Training data found: {train_file} ({size_mb:.1f} MB)")
else:
    print(f"ERROR: Training data NOT found at {train_file}")
    print(f"Please upload train.jsonl and val.jsonl to Google Drive > legal-contract-llm folder")

---
## CELL 4: Load Model with 4-bit Quantization
Load Qwen2.5-3B with QLoRA (4-bit) to fit in T4's 16GB VRAM.
This compresses the 3B model from ~6GB to ~1.7GB.

In [ ]:
from unsloth import FastLanguageModel

# Load model - 4-bit quantization makes it fit on T4
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=2048,    # Max tokens per example
    dtype=None,             # Auto-detect (float16 for T4)
    load_in_4bit=True,      # QLoRA: 4-bit quantization
)

print(f"\nModel loaded successfully!")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

---
## CELL 5: Add LoRA Adapters
LoRA adds small trainable layers on top of the frozen model.
We only train ~1.5% of parameters (the LoRA layers), not the full model.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                    # LoRA rank (capacity of adapters)
    target_modules=[         # Which layers to add LoRA to
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention layers
        "gate_proj", "up_proj", "down_proj",      # MLP layers
    ],
    lora_alpha=32,           # Scaling factor (2x rank)
    lora_dropout=0,          # No dropout (Unsloth optimized)
    bias="none",             # No bias in LoRA
    use_gradient_checkpointing="unsloth",  # Saves 60% VRAM!
    random_state=42,
)

# Show how many parameters we're actually training
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

---
## CELL 6: Load Training Data
Load our formatted CUAD dataset from Google Drive.

In [ ]:
import json
from datasets import Dataset

def load_jsonl(filepath):
    """Load JSONL file into a list of dicts."""
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return data

# Load training and validation data
train_data = load_jsonl(train_file)
val_data = load_jsonl(val_file)

print(f"Training examples: {len(train_data):,}")
print(f"Validation examples: {len(val_data):,}")

# Convert messages to the text format Qwen expects
def format_for_training(example):
    """Apply the chat template to convert messages to text."""
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

# Create HuggingFace datasets
train_dataset = Dataset.from_list(train_data).map(format_for_training)
val_dataset = Dataset.from_list(val_data).map(format_for_training)

print(f"\nDataset ready!")
print(f"Sample text length: {len(train_dataset[0]['text']):,} characters")
print(f"\nFirst 300 chars of a training example:")
print(train_dataset[0]['text'][:300])

---
## CELL 7: Configure Training
Set up the training parameters.
- 200 steps = ~50 min on T4 (good starting point)
- 500 steps = ~2 hours on T4 (better quality)
- Increase max_steps for better results

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=True,           # Pack short examples together = faster
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,   # Effective batch = 8
        warmup_steps=10,
        max_steps=200,                   # START WITH 200 (~50 min)
        # max_steps=500,                 # UNCOMMENT for better quality (~2 hrs)
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        eval_steps=50,
        eval_strategy="steps",
        save_strategy="steps",
        save_steps=50,                   # Save checkpoint every 50 steps
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

print("Trainer configured!")
print(f"Max steps: {trainer.args.max_steps}")
print(f"Batch size: {trainer.args.per_device_train_batch_size}")
print(f"Gradient accumulation: {trainer.args.gradient_accumulation_steps}")
print(f"Effective batch size: {trainer.args.per_device_train_batch_size * trainer.args.gradient_accumulation_steps}")

---
## CELL 8: TRAIN! (This takes ~50 min for 200 steps)
This is where the actual fine-tuning happens.
Watch the loss value — it should DECREASE over time.

**DO NOT close this tab while training!**

In [ ]:
# Check GPU memory before training
gpu_stats = torch.cuda.get_device_properties(0)
start_mem = torch.cuda.memory_allocated() / 1024**3
print(f"GPU: {gpu_stats.name} ({gpu_stats.total_mem / 1024**3:.1f} GB total)")
print(f"Memory used before training: {start_mem:.1f} GB")
print(f"\nStarting training... This will take ~50 minutes for 200 steps.")
print(f"Watch the 'loss' value - it should decrease over time.\n")

# TRAIN!
trainer_stats = trainer.train()

# Print results
print(f"\n{'='*50}")
print(f"TRAINING COMPLETE!")
print(f"{'='*50}")
print(f"Steps:        {trainer_stats.global_step}")
print(f"Final Loss:   {trainer_stats.metrics['train_loss']:.4f}")
print(f"Training Time: {trainer_stats.metrics['train_runtime']:.0f} seconds ({trainer_stats.metrics['train_runtime']/60:.1f} minutes)")
print(f"Peak Memory:  {torch.cuda.max_memory_allocated() / 1024**3:.1f} GB")

---
## CELL 9: Test the Model
Let's see if the fine-tuned model can analyze contract clauses!

In [ ]:
# Switch to inference mode (faster generation)
FastLanguageModel.for_inference(model)

# Test contracts
test_clauses = [
    """The Employee agrees that during the term of employment and for a period 
    of two (2) years following termination, Employee shall not directly or 
    indirectly engage in any business that competes with the Company within 
    a 50-mile radius of any Company office.""",
    
    """This Agreement shall be governed by and construed in accordance with 
    the laws of the State of Delaware, without regard to its conflict of 
    laws principles.""",
    
    """IN NO EVENT SHALL EITHER PARTY'S TOTAL LIABILITY UNDER THIS AGREEMENT 
    EXCEED THE TOTAL FEES PAID BY CUSTOMER DURING THE TWELVE (12) MONTH 
    PERIOD IMMEDIATELY PRECEDING THE EVENT GIVING RISE TO SUCH LIABILITY.""",
]

system_prompt = """You are a legal contract clause analyzer. When given a contract clause, you must:
1. Identify the clause type
2. Extract the key clause text and important terms
3. Assess the risk level (HIGH, MEDIUM, or LOW)
4. Provide a plain English explanation
Respond in JSON format."""

for i, clause in enumerate(test_clauses, 1):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Analyze the following contract clause and identify any relevant legal provisions:\n\n{clause}"},
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")
    
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=512,
        temperature=0.1,
        do_sample=True,
    )
    
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    
    print(f"\n{'='*60}")
    print(f"TEST {i}:")
    print(f"Input: {clause[:100]}...")
    print(f"\nModel Output:")
    print(response)
    print(f"{'='*60}")

---
## CELL 10: Save Model to Google Drive
Save the LoRA adapter (~50-100 MB) to Google Drive so you can download it to your laptop.

In [ ]:
# Save LoRA adapter to Google Drive
save_dir = '/content/drive/MyDrive/legal-contract-llm/model'
os.makedirs(save_dir, exist_ok=True)

print("Saving LoRA adapter to Google Drive...")
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

# Check saved size
total_size = 0
for f in os.listdir(save_dir):
    fpath = os.path.join(save_dir, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath) / (1024*1024)
        total_size += size
        print(f"  {f}: {size:.1f} MB")

print(f"\nTotal saved: {total_size:.1f} MB")
print(f"Location: {save_dir}")
print(f"\nYou can now download this from Google Drive to your laptop!")

---
## CELL 11 (OPTIONAL): Export to GGUF for Ollama
Export the model to GGUF format so you can run it locally with Ollama.
This creates a ~2.1 GB file.

In [ ]:
# OPTIONAL: Export to GGUF for local inference with Ollama
# Uncomment the lines below if you want GGUF export

# gguf_dir = '/content/drive/MyDrive/legal-contract-llm/gguf'
# os.makedirs(gguf_dir, exist_ok=True)
# print("Exporting to GGUF (this takes ~5 minutes)...")
# model.save_pretrained_gguf(
#     gguf_dir,
#     tokenizer,
#     quantization_method="q4_k_m"   # Best quality/size balance (~2.1 GB)
# )
# print(f"GGUF exported to: {gguf_dir}")

print("To export GGUF, uncomment the code above and run this cell.")
print("GGUF is needed only if you want to run the model locally with Ollama.")

---
## CELL 12 (OPTIONAL): Push to HuggingFace Hub
Publish your model so anyone can use it and hiring managers can see it.

In [ ]:
# OPTIONAL: Push to HuggingFace Hub
# Uncomment and add your HuggingFace token

# HF_TOKEN = "hf_YOUR_TOKEN_HERE"  # Get from https://huggingface.co/settings/tokens
# model.push_to_hub("YOUR_USERNAME/legal-contract-clause-analyzer", token=HF_TOKEN)
# tokenizer.push_to_hub("YOUR_USERNAME/legal-contract-clause-analyzer", token=HF_TOKEN)
# print("Model pushed to HuggingFace Hub!")

print("To push to HuggingFace, uncomment the code above.")
print("You'll need a HuggingFace account and access token.")

---
## DONE!

**What you've accomplished:**
1. Loaded Qwen2.5-3B with 4-bit quantization
2. Added LoRA adapters (only ~1.5% trainable parameters)
3. Trained on 16,270 legal contract examples
4. Model can now classify 41 clause types + explain in plain English
5. Saved to Google Drive for download

**Next steps:**
- Download the model from Google Drive to your laptop
- Build the FastAPI + Streamlit demo
- Run evaluation on the test set